# RQ3: SILAM-0.1° and SILAM-DS compared with observed Betula pollen

This notebook compares the onset of the Betula pollen season (5 % of the seasonal sum) from SILAM-0.1° (FMI, 2023 to 2026) and SILAM-DS (RIVM, 2025 and 2026) with the observations in Leiden and Helmond, and with the GDD onsets from RQ2.

**Data (not included in this repository):** SILAM-0.1° runs (`silam_cut_netherlands_YYYYMMDD.nc4`), pollen counts of Leiden (LUMC) and Helmond (Elkerliek), SILAM-DS daily files and the GDD onsets from RQ2.

## Step 1: Setup
Load the packages.

In [ ]:
# pip install netCDF4 openpyxl
import io
import os
import re
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from netCDF4 import Dataset

warnings.filterwarnings("ignore")

## Step 2: Configuration
Set the data paths, the study period and the evaluation settings.

In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("output_rq3")
OUT_DIR.mkdir(exist_ok=True)

SILAM_DIR = DATA_DIR / "silam"                  # SILAM-0.1° runs (.nc4)
LEIDEN_XLSX = DATA_DIR / "pollen_leiden.xlsx"   # LUMC pollen counts
HELMOND_XLSX = DATA_DIR / "pollen_helmond.xlsx" # Elkerliek pollen counts
DS_DIR = DATA_DIR / "silam_ds"                  # SILAM-DS daily files (RIVM)
GDD_FILE = DATA_DIR / "gdd_onsets_loyo.csv"     # GDD onsets from RQ2 (columns: station, year, onset)

# Unpack zipped SILAM runs if needed
for z in sorted(DATA_DIR.glob("silam_slim_*.zip")):
    with zipfile.ZipFile(z) as f:
        f.extractall(SILAM_DIR)

YEARS = [2023, 2024, 2025, 2026]
SEASON_START = "03-01"
SEASON_END = "05-31"
HORIZONS = [0, 1]      # h = 0: hours 1 to 24 of run S = t, h = 1: hours 25 to 48 of run S = t - 1
WINDOWS = [1, 3, 5]    # window width in grid cells (1x1, 3x3, 5x5)
MAIN_WINDOW = 3        # used for tables and figures (3x3 cells, about 21 x 33 km)
ONSET_FRAC = 0.05
END_FRAC = 0.95
VAR = "cnc_POLLEN_BIRCH_m22"

STATIONS = {
    "Leiden": {"lat": 52.166031, "lon": 4.478504},   # LUMC
    "Helmond": {"lat": 51.482303, "lon": 5.642436},  # Elkerliek
}

## Step 3: Check the files and the grid
Count the daily runs per year and find the grid cell of each pollen station.

In [ ]:
pat = re.compile(r"silam_cut_netherlands_(\d{8})\.nc4$")
files = {}
for p in sorted(SILAM_DIR.rglob("silam_cut_netherlands_*.nc4")):
    m = pat.search(p.name)
    if m:
        files[pd.Timestamp(m.group(1))] = p

if not files:
    raise FileNotFoundError(f"No SILAM files found in {SILAM_DIR.resolve()}")

print(f"{len(files)} runs found")
for y in YEARS:
    expected = pd.date_range(f"{y}-{SEASON_START}", f"{y}-{SEASON_END}")
    have = [d for d in expected if d in files]
    print(f"  {y}: {len(have)} of {len(expected)} days")

with Dataset(files[min(files)]) as ds:
    lat = np.asarray(ds["lat"][:].data, float)
    lon = np.asarray(ds["lon"][:].data, float)

R = max(WINDOWS) // 2
CELLS = {}
for name, c in STATIONS.items():
    i = int(np.argmin(np.abs(lat - c["lat"])))
    j = int(np.argmin(np.abs(lon - c["lon"])))
    if not (R <= i < len(lat) - R and R <= j < len(lon) - R):
        raise ValueError(f"{name}: window of {max(WINDOWS)} cells falls outside the grid")
    CELLS[name] = (i, j)
    print(f"  {name:8s} cell (lat {lat[i]:.2f}, lon {lon[j]:.2f})")

dy_km = np.diff(lat).mean() * 111.2
dx_km = np.diff(lon).mean() * 111.32 * np.cos(np.radians(52.0))
print(f"Cell size: {dy_km:.1f} km (N-S) x {dx_km:.1f} km (E-W)")

## Step 4: Read hours 1 to 48 of each run
Store the hourly mean over each window per run and station, cached in `hourly_cache.pkl`.

In [ ]:
MAXH = 24 * (max(HORIZONS) + 1)   # 48 hours per run
CACHE = OUT_DIR / "hourly_cache.pkl"

def extract_run(path, run_date):
    with Dataset(path) as ds:
        la = np.asarray(ds["lat"][:].data, float)
        lo = np.asarray(ds["lon"][:].data, float)
        if not (np.allclose(la, lat) and np.allclose(lo, lon)):
            raise ValueError(f"Different grid in {path.name}")
        t = ds["time"]
        start = pd.Timestamp(t.units.split("since")[1].strip()[:10])
        if start != run_date:
            raise ValueError(f"{path.name}: time axis starts on {start.date()}, not on {run_date.date()}")
        secs = np.asarray(t[:MAXH].data, float)
        var = ds[VAR]
        res = {}
        for name, (i, j) in CELLS.items():
            block = np.ma.filled(var[:MAXH, 0, i-R:i+R+1, j-R:j+R+1], np.nan)
            for w in WINDOWS:
                r = w // 2
                res[(name, w)] = np.nanmean(block[:, R-r:R+r+1, R-r:R+r+1], axis=(1, 2))
    return secs, res

run_dates = [d for d in sorted(files) if d.year in YEARS]
# The cache is only valid for the same settings and runs
SIG = dict(years=list(YEARS), windows=list(WINDOWS), maxh=MAXH, var=VAR,
           stations={k: (v["lat"], v["lon"]) for k, v in STATIONS.items()},
           runs=[d.strftime("%Y%m%d") for d in run_dates])
hourly = None
if CACHE.exists():
    cached = pd.read_pickle(CACHE)
    if isinstance(cached, dict) and cached.get("sig") == SIG:
        hourly = cached["df"]
if hourly is None:
    rows = []
    for d in run_dates:
        secs, res = extract_run(files[d], d)
        valid = d + pd.to_timedelta(secs, unit="s")
        lead = (secs / 3600).round().astype(int)
        for (name, w), vals in res.items():
            rows.append(pd.DataFrame({"station": name, "window": w, "run": d, "lead_h": lead,
                                      "valid_utc": valid, "value": vals}))
    hourly = pd.concat(rows, ignore_index=True)
    # hours 1 to 24 belong to day S (h = 0), hours 25 to 48 to day S + 1 (h = 1)
    hourly["h"] = (hourly["lead_h"] - 1) // 24
    hourly["day"] = (hourly["valid_utc"] - pd.Timedelta(hours=1)).dt.normalize()
    pd.to_pickle({"sig": SIG, "df": hourly}, CACHE)
print(f"{len(hourly)} hourly values")

## Step 5: Daily series per lead time
Average the hourly values per day, using only days with all 24 hours from the same run.

In [ ]:
sel = hourly[hourly["h"].isin(HORIZONS)]
g = (sel.groupby(["station", "window", "h", "day"])["value"]
        .agg(["mean", "count"]).reset_index())
daily = g[g["count"] == 24].rename(columns={"mean": "silam"}).drop(columns="count")
daily["year"] = daily["day"].dt.year
daily.head()

## Step 6: Read the pollen counts
Read the daily Betula counts, taking the year of each Helmond sheet from the sheet name because the date headers are unreliable.

In [ ]:
def load_leiden(path):
    df = pd.read_excel(path)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    s = pd.to_numeric(df["Betula"], errors="coerce")
    s = pd.Series(s.values, index=df["date"])
    s = s[~s.index.duplicated()]
    return s.sort_index()

def load_helmond(path, years):
    xl = pd.ExcelFile(path)
    parts = []
    for sheet in xl.sheet_names:
        if not sheet.strip().isdigit() or int(sheet.strip()) not in years:
            continue
        Y = int(sheet.strip())
        raw = xl.parse(sheet, header=None)
        hdr = pd.to_datetime(raw.iloc[0, 1:], errors="coerce")
        dates = []
        for k, d in enumerate(hdr):
            if pd.isna(d):
                dates.append(pd.NaT)
                continue
            yy = Y - 1 if (d.month == 12 and k < 60) else Y
            try:
                dates.append(pd.Timestamp(yy, d.month, d.day))
            except ValueError:
                dates.append(pd.NaT)
        # first Betula row that contains counts
        rows = [i for i in raw.index if str(raw.iat[i, 0]).strip() == "Betula"]
        row = next((i for i in rows if pd.to_numeric(raw.iloc[i, 1:], errors="coerce").sum() > 0), None)
        if row is None:
            print(f"Helmond {Y}: no Betula counts")
            continue
        # empty cell = 0, text (instrument failure) = missing
        cells_ = raw.iloc[row, 1:].tolist()
        vals = [np.nan if isinstance(c, str) else (0.0 if pd.isna(c) else float(c)) for c in cells_]
        s = pd.Series(vals, index=pd.DatetimeIndex(dates))
        parts.append(s[s.index.notna()])
    out = pd.concat(parts).sort_index()
    return out[~out.index.duplicated()]

OBS = {
    "Leiden": load_leiden(LEIDEN_XLSX),
    "Helmond": load_helmond(HELMOND_XLSX, YEARS),
}
for name, s in OBS.items():
    print(f"{name}: {s.index.min().date()} to {s.index.max().date()}, {s.notna().sum()} days")

## Step 7: Alignment check
Compute the rank correlation between model and observations for shifts of −2 to +2 days to detect date errors.

In [ ]:
def window_series(station, year, w, h):
    idx = pd.date_range(f"{year}-{SEASON_START}", f"{year}-{SEASON_END}")
    d = daily[(daily["station"] == station) & (daily["window"] == w) & (daily["h"] == h) & (daily["year"] == year)]
    return pd.Series(d["silam"].values, index=d["day"]).reindex(idx)

rows = []
for st in STATIONS:
    for y in YEARS:
        mod = window_series(st, y, MAIN_WINDOW, 0)
        obs = OBS[st].reindex(mod.index)
        r = {"station": st, "year": y}
        for lag in range(-2, 3):
            r[f"lag {lag:+d}"] = mod.corr(obs.shift(-lag), method="spearman")
        lagcols = [c for c in r if c.startswith("lag")]
        r["best lag"] = max(lagcols, key=lambda c: -9 if pd.isna(r[c]) else r[c])
        rows.append(r)

align = pd.DataFrame(rows).set_index(["station", "year"])
align.round(2)

## Step 8: Onset, peak, volume and bias
Compute the onset, peak and seasonal sum per station, year, window and lead time, and the deviation from the observations.

In [ ]:
def season_stats(s):
    s = s.fillna(0.0)
    tot = float(s.sum())
    if tot <= 0:
        return dict(onset=pd.NaT, peak=pd.NaT, end=pd.NaT, spin=0.0)
    c = (s.cumsum() / tot).values
    return dict(onset=s.index[int(np.argmax(c >= ONSET_FRAC))],
                peak=s.idxmax(),
                end=s.index[int(np.argmax(c >= END_FRAC))],
                spin=tot)

def dev(a, b):
    return (a - b).days if pd.notna(a) and pd.notna(b) else np.nan

rows = []
for st in STATIONS:
    for y in YEARS:
        idx = pd.date_range(f"{y}-{SEASON_START}", f"{y}-{SEASON_END}")
        obs = OBS[st].reindex(idx)
        so = season_stats(obs)
        for w in WINDOWS:
            for h in HORIZONS:
                mod = window_series(st, y, w, h)
                sm = season_stats(mod)
                both = pd.concat([mod, obs], axis=1, keys=["m", "o"]).dropna()
                rows.append({
                    "station": st, "year": y, "window": w, "h": h,
                    "obs_onset": so["onset"], "mod_onset": sm["onset"],
                    "onset_dev_d": dev(sm["onset"], so["onset"]),
                    "obs_peak": so["peak"], "mod_peak": sm["peak"],
                    "peak_dev_d": dev(sm["peak"], so["peak"]),
                    "obs_end": so["end"], "mod_end": sm["end"],
                    "obs_spin": so["spin"], "mod_spin": sm["spin"],
                    "spin_ratio": sm["spin"] / so["spin"] if so["spin"] > 0 else np.nan,
                    "MB": (both["m"] - both["o"]).mean() if len(both) else np.nan,
                    "RMSE": float(np.sqrt(((both["m"] - both["o"]) ** 2).mean())) if len(both) else np.nan,
                    "r": both["m"].corr(both["o"]) if len(both) > 2 else np.nan,
                    "n_days": len(both),
                    "n_model_missing": int(mod.isna().sum()),
                    "n_obs_missing": int(obs.isna().sum()),
                })
results = pd.DataFrame(rows)

main = results[results["window"] == MAIN_WINDOW].copy()
cols = ["station", "year", "h", "obs_onset", "mod_onset", "onset_dev_d", "peak_dev_d", "spin_ratio", "MB", "RMSE", "r"]
main[cols].round(2)

## Step 9: Sensitivity to window size
Show the onset deviation per window size and lead time.

In [ ]:
sens = results.pivot_table(index=["station", "year"], columns=["h", "window"], values="onset_dev_d")
sens.columns = [f"h{h}, {w}x{w}" for h, w in sens.columns]
sens

## Step 10: Cumulative curves
Plot the normalised cumulative curves of observations and SILAM-0.1° per station and year.

In [ ]:
FIG_H = 0
fig, axes = plt.subplots(len(STATIONS), len(YEARS), figsize=(max(4.2 * len(YEARS), 7.5), 3.2 * len(STATIONS)),
                         sharey=True, squeeze=False)
for r_i, st in enumerate(STATIONS):
    for c_i, y in enumerate(YEARS):
        ax = axes[r_i][c_i]
        idx = pd.date_range(f"{y}-{SEASON_START}", f"{y}-{SEASON_END}")
        obs = OBS[st].reindex(idx).fillna(0)
        mod = window_series(st, y, MAIN_WINDOW, FIG_H).fillna(0)
        if obs.sum() > 0:
            ax.plot(idx, obs.cumsum() / obs.sum(), color="black", lw=2, label="observed")
        if mod.sum() > 0:
            ax.plot(idx, mod.cumsum() / mod.sum(), color="tab:red", lw=1.8, label=f"SILAM-0.1° (h = {FIG_H})")
        ax.axhline(ONSET_FRAC, color="grey", lw=0.8, ls=":")
        row = results[(results.station == st) & (results.year == y) & (results.window == MAIN_WINDOW) & (results.h == FIG_H)].iloc[0]
        ax.set_title(f"{st} {y}\nonset deviation {row['onset_dev_d']:+.0f} d, SPIn ratio {row['spin_ratio']:.2f}", fontsize=9)
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
        if c_i == 0:
            ax.set_ylabel("cumulative fraction")
        if r_i == 0 and c_i == 0:
            ax.legend(fontsize=8, loc="upper left")
fig.suptitle("Cumulative Betula pollen, observed and SILAM-0.1°", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_cumulative_silam01.png", dpi=200)
plt.show()

## Step 11: Read SILAM-DS
Read the SILAM-DS daily files (total, local and background component per point).

In [ ]:
DS_FILES = {
    "daily_2025": DS_DIR / "silam_ds_daily_2025.csv",
    "daily_2026": DS_DIR / "silam_ds_daily_2026.csv",
    "daily_ens_2026": DS_DIR / "silam_ds_daily_ens_2026.csv",
}

def read_ds(key):
    return pd.read_csv(DS_FILES[key], parse_dates=["dateUTC"]).set_index("dateUTC")

DS = {2025: read_ds("daily_2025"), 2026: read_ds("daily_2026")}
DS_ENS = read_ds("daily_ens_2026")

for y, df in DS.items():
    print(f"SILAM-DS {y}: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")

## Step 12: Onset of observations, SILAM-0.1° and SILAM-DS
Compute the onset of the three series within the same period, together with the onset of the local and background components of SILAM-DS.

In [ ]:
DS_COL = {"Leiden": "lumc", "Helmond": "elk"}
CMP_WINDOW = 1   # station cell, because SILAM-DS is a point value
CMP_H = 0

rows = []
CMP = {}
for st in STATIONS:
    for y in sorted(DS):
        df = DS[y]
        win = df.index
        ds_series = df[DS_COL[st] + "Total"]
        obs = OBS[st].reindex(win)
        std = window_series(st, y, CMP_WINDOW, CMP_H).reindex(win)
        CMP[(st, y)] = pd.DataFrame({"observed": obs, "silam01": std, "silam_ds": ds_series})
        so, ss, sd_ = season_stats(obs), season_stats(std), season_stats(ds_series)
        loc = season_stats(df[DS_COL[st] + "Local"])
        bg = season_stats(df[DS_COL[st] + "Bg"])
        pair = pd.concat([std, ds_series], axis=1).dropna()
        rows.append({
            "station": st, "year": y,
            "period": f"{win.min():%d %b} - {win.max():%d %b}",
            "obs_onset": so["onset"], "std_onset": ss["onset"], "ds_onset": sd_["onset"],
            "std_dev_d": dev(ss["onset"], so["onset"]), "ds_dev_d": dev(sd_["onset"], so["onset"]),
            "ds_local_onset": loc["onset"], "ds_bg_onset": bg["onset"],
            "obs_spin": so["spin"], "std_spin": ss["spin"], "ds_spin": sd_["spin"],
            "obs_peak": so["peak"], "std_peak": ss["peak"], "ds_peak": sd_["peak"],
            "r_std_ds": pair.iloc[:, 0].corr(pair.iloc[:, 1]),
        })
ds_vs_std = pd.DataFrame(rows)
show = ["station", "year", "period", "obs_onset", "std_onset", "ds_onset", "std_dev_d", "ds_dev_d",
        "ds_local_onset", "ds_bg_onset", "r_std_ds"]
ds_vs_std[show].round(2)

## Step 13: Enschede (model only)
Compute the SILAM-DS onset for Enschede in 2026, where no pollen counts are available.

In [ ]:
for part in ["Total", "Local", "Bg"]:
    s = season_stats(DS_ENS["ens" + part])
    onset = f"{s['onset']:%d %b %Y}" if pd.notna(s["onset"]) else "none"
    print(f"{part:5s}: onset {onset}, seasonal sum {s['spin']:.0f}")

## Step 14: Figures of observations, SILAM-0.1° and SILAM-DS
Plot the normalised cumulative curves and the daily concentrations, with dotted lines at the onset.

In [ ]:
def three_line_figure(kind):
    keys = [(st, y) for y in sorted(DS) for st in STATIONS]
    fig, axes = plt.subplots(len(DS), len(STATIONS), figsize=(11, 3.6 * len(DS)), squeeze=False)
    col = {"observed": "black", "silam01": "tab:red", "silam_ds": "tab:blue"}
    lab = {"observed": "observed", "silam01": f"SILAM-0.1° ({CMP_WINDOW}x{CMP_WINDOW})", "silam_ds": "SILAM-DS"}
    for k, (st, y) in enumerate(keys):
        ax = axes[sorted(DS).index(y)][list(STATIONS).index(st)]
        d = CMP[(st, y)]
        for name in ["observed", "silam01", "silam_ds"]:
            s = d[name].fillna(0)
            if s.sum() <= 0:
                continue
            on = season_stats(s)["onset"]
            if kind == "cum":
                ax.plot(s.index, s.cumsum() / s.sum(), color=col[name], lw=2 if name == "observed" else 1.6, label=lab[name])
            else:
                ax.plot(s.index, s, color=col[name], lw=2 if name == "observed" else 1.4, label=lab[name])
            ax.axvline(on, color=col[name], ls=":", lw=1)
        if kind == "cum":
            ax.axhline(ONSET_FRAC, color="grey", lw=0.8, ls=":")
            ax.set_ylabel("cumulative fraction")
        else:
            ax.set_ylabel("pollen grains m$^{-3}$ (daily mean)")
        ax.set_title(f"{st} {y}", fontsize=10)
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)
        if k == 0:
            ax.legend(fontsize=7, loc="upper left")
    fig.suptitle("Normalised cumulative curves" if kind == "cum" else "Daily concentrations", fontsize=11)
    fig.tight_layout()
    return fig

fig_a = three_line_figure("cum")
fig_a.savefig(OUT_DIR / "fig_three_series_cumulative.png", dpi=200)
plt.show()
fig_b = three_line_figure("daily")
fig_b.savefig(OUT_DIR / "fig_three_series_daily.png", dpi=200)
plt.show()

## Step 15: GDD onsets
Read the leave-one-year-out GDD onsets from RQ2 for the comparison with SILAM.

In [ ]:
GDD_ONSET = {}
if GDD_FILE.exists():
    g = pd.read_csv(GDD_FILE, parse_dates=["onset"])
    for _, r in g.iterrows():
        GDD_ONSET[(r["station"], int(r["year"]))] = r["onset"]
print(f"{len(GDD_ONSET)} GDD onsets")

## Step 16: Overview per case
Plot the onset deviation of SILAM-0.1°, SILAM-DS and GDD for each station and year.

In [ ]:
rows = []
for st in STATIONS:
    for y in YEARS:
        r0 = results[(results.station == st) & (results.year == y) & (results.window == MAIN_WINDOW) & (results.h == 0)].iloc[0]
        row = {"station": st, "year": y, "obs_onset": r0["obs_onset"], "silam01_dev": r0["onset_dev_d"]}
        ds_row = ds_vs_std[(ds_vs_std.station == st) & (ds_vs_std.year == y)]
        row["silam_ds_dev"] = float(ds_row["ds_dev_d"].iloc[0]) if len(ds_row) else np.nan
        gd = GDD_ONSET.get((st, y))
        row["gdd_onset"] = gd
        row["gdd_dev"] = dev(gd, r0["obs_onset"]) if gd is not None else np.nan
        rows.append(row)
summary = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4))
labels = [f"{r.station[:3]} {r.year}" for r in summary.itertuples()]
x = np.arange(len(summary))
ax.axhline(0, color="black", lw=1)
ax.scatter(x - 0.12, summary["silam01_dev"], color="tab:red", s=55, label="SILAM-0.1°", zorder=3)
ax.scatter(x, summary["silam_ds_dev"], color="tab:blue", marker="s", s=55, label="SILAM-DS", zorder=3)
if summary["gdd_dev"].notna().any():
    ax.scatter(x + 0.12, summary["gdd_dev"], color="tab:green", marker="^", s=60, label="GDD (LOYO)", zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.set_ylabel("onset deviation (days, model - observed)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_onset_deviation_overview.png", dpi=200)
plt.show()
summary.round(1)

## Step 17: Save
Save all tables to one Excel file.

In [ ]:
with pd.ExcelWriter(OUT_DIR / "rq3_results.xlsx", engine="openpyxl") as w:
    results.to_excel(w, sheet_name="results", index=False)
    sens.to_excel(w, sheet_name="window_sensitivity")
    align.to_excel(w, sheet_name="alignment")
    ds_vs_std.to_excel(w, sheet_name="silam_ds_vs_silam01", index=False)
    summary.to_excel(w, sheet_name="summary", index=False)
    daily.to_excel(w, sheet_name="silam_daily", index=False)